# H10 — Scaling without a quantum-advantage claim
This notebook runs locally and submits no provider jobs. Compare identical synthetic slot-assignment inputs using enumeration and SciPy/HiGHS MILP. CP-SAT is not measured. A search limit is a censored observation, not proof that all classical methods fail.

In [1]:
from pathlib import Path
import sys, json
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "backend/app/scaling.py").exists())
sys.path[:0] = [str(root), str(root / "backend")]
from app.scaling import ScalingCase, make_problem, enumerate_problem, solve_milp, benchmark
from app.research_api import read_report, scaling_view
from IPython.display import display, HTML
report = read_report("scaling", root / "shared/examples")
report["environment"], report["budget"]

({'python': '3.12.4',
  'numpy': '2.5.4',
  'scipy': '1.18.1',
  'platform': 'macOS-27.2.1-arm64-arm-64bit',
  'machine': 'arm64',
  'processor': 'arm'},
 {'repeats': 3, 'max_evaluations': 65536, 'time_limit_s': 0.5})

Each mission chooses one slot. Slots have declared capacities; preferences are seeded integers. This structured problem may remain easy for MILP even when its Cartesian search space is large.

In [2]:
problem = make_problem(ScalingCase(missions=3, options=4, seed=17))
problem

{'missions': 3,
 'options': 4,
 'seed': 17,
 'utilities': [[75, 85, 11, 17], [46, 56, 77, 37], [4, 22, 46, 39]],
 'capacities': [1, 1, 1, 1],
 'fingerprint': 'd2cc378e997f02ae511188ec2033ed90704f97b54039430eafacec5af6fd9808',
 'combinations': 64,
 'logical_qubits': 6,
 'one_hot_variables': 12}

In [3]:
exact = enumerate_problem(problem, time_limit_s=5)
milp = solve_milp(problem, time_limit_s=5)
assert exact["score"] == milp["score"]
{"exact": exact, "milp": milp}

{'exact': {'status': 'optimal',
  'complete': True,
  'evaluated': 64,
  'feasible_count': 24,
  'selection': (1, 2, 3),
  'score': 201,
  'elapsed_s': 7.112499588401988e-05},
 'milp': {'status': 'optimal',
  'selection': (1, 2, 3),
  'score': 201,
  'validation': {'feasible': True, 'score': 201, 'occupancy': [0, 1, 1, 1]},
  'mip_gap': 0.0,
  'elapsed_s': 0.007663791999220848,
  'message': 'Optimization terminated successfully. (HiGHS Status 7: Optimal)'}}

In [4]:
rows = "".join(f"<tr><td>{p['missions']}</td><td>{p['options']}</td><td>{p['combinations']}</td><td>{p['logical_qubits']}</td><td>{p['exact_median_s']:.6f}</td><td>{p['exact_runs'][-1]['status']}</td><td>{p['milp_median_s']:.6f}</td><td>{p['milp_runs'][-1]['status']}</td></tr>" for p in report['points'])
display(HTML('<table><tr><th>Missions</th><th>Options</th><th>Plans</th><th>Logical qubits</th><th>Exact seconds</th><th>Status</th><th>MILP seconds</th><th>Status</th></tr>'+rows+'</table>'))

Missions,Options,Plans,Logical qubits,Exact seconds,Status,MILP seconds,Status
3,2,8,3,0.000011,optimal,0.000582,optimal
6,2,64,6,0.000069,optimal,0.004282,optimal
10,2,1024,10,0.001330,optimal,0.003343,optimal
16,2,65536,16,0.105353,optimal,0.002741,optimal
24,2,16777216,24,0.138219,limit_reached,0.002609,optimal
3,4,64,6,0.000062,optimal,0.004304,optimal
6,4,4096,12,0.004609,optimal,0.003788,optimal
10,4,1048576,20,0.087903,limit_reached,0.003577,optimal
16,4,4294967296,32,0.109151,limit_reached,0.003263,optimal
24,4,281474976710656,48,0.144519,limit_reached,0.003029,optimal


In [5]:
view = scaling_view(report)
colors = ["#22d3ee", "#a78bfa", "#fbbf24", "#fb7185", "#34d399", "#94a3b8"]
for plot in view["plots"]:
    marks = []
    for i, series in enumerate(plot["series"]):
        for point in series["points"]:
            marks.append(f'<circle cx="{40+point["x"]*5}" cy="{20+point["y"]*2}" r="4" fill="{colors[i%len(colors)]}"><title>{series["name"]}: {point["missions"]} missions, {point["options"]} options, {point["value"]} ({point["kind"]})</title></circle>')
    display(HTML(f'<h3>{plot["metric"]} (log vertical axis)</h3><svg viewBox="0 0 600 250" width="600"><path d="M40 20V220H550" stroke="gray" fill="none"/>'+''.join(marks)+'</svg>'))

In [6]:
# A bounded fresh measurement; timing differs between machines/runs.
fresh = benchmark((ScalingCase(missions=3, options=4), ScalingCase(missions=6, options=4)), repeats=2, max_evaluations=10000, time_limit_s=0.5)
[(p["missions"], p["exact_median_s"], p["milp_median_s"]) for p in fresh["points"]]

[(3, 7.322899546124972e-05, 0.004467562001082115),
 (6, 0.004867854000622174, 0.004003791502327658)]

In [7]:
report["projections"]

[{'missions': 8,
  'options': 2,
  'combinations': '256',
  'estimated_exact_s': 0.0003493439871817827,
  'measurement_kind': 'extrapolated',
  'source_fingerprint': '159713d488998e161d9ad6c7cef2cd9053c677352ae2b3263d840b9ab4dac3e0'},
 {'missions': 12,
  'options': 2,
  'combinations': '4096',
  'estimated_exact_s': 0.0055895037949085236,
  'measurement_kind': 'extrapolated',
  'source_fingerprint': '159713d488998e161d9ad6c7cef2cd9053c677352ae2b3263d840b9ab4dac3e0'},
 {'missions': 16,
  'options': 2,
  'combinations': '65536',
  'estimated_exact_s': 0.08943206071853638,
  'measurement_kind': 'extrapolated',
  'source_fingerprint': '159713d488998e161d9ad6c7cef2cd9053c677352ae2b3263d840b9ab4dac3e0'},
 {'missions': 24,
  'options': 2,
  'combinations': '16777216',
  'estimated_exact_s': 22.894607543945312,
  'measurement_kind': 'extrapolated',
  'source_fingerprint': '159713d488998e161d9ad6c7cef2cd9053c677352ae2b3263d840b9ab4dac3e0'},
 {'missions': 32,
  'options': 2,
  'combinations': '4

Extrapolations multiply a measured small-case cost per assignment by the larger Cartesian count. They are not measurements, solver lower bounds, or MILP predictions. Logical qubit counts describe compact registers only; they exclude routing, ancillas, error correction, preparation and optimization costs. Larger compact Hamiltonians can require many gates. At three missions classical computation is sufficient.